# Day 31 Exercises: PyTorch Basics

This notebook covers fundamental concepts in PyTorch: Tensors, Autograd, nn.Module, DataLoader, and Optimizers.

## 1. Tensors and Autograd

In this section, we create basic PyTorch tensors and demonstrate how `autograd` automatically calculates gradients (derivatives) during the `.backward()` pass, which is essential for training neural networks.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Create a tensor with requires_grad=True
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
print("Original Tensor:\n", x)

# Perform some operations
y = x ** 2 + 5
print("\nResult of y = x^2 + 5:\n", y)

# Compute the mean and backpropagate
out = y.mean()
out.backward()

print("\nGradients (dy/dx):\n", x.grad)


Original Tensor:
 tensor([[1., 2.],
        [3., 4.]], requires_grad=True)

Result of y = x^2 + 5:
 tensor([[ 6.,  9.],
        [14., 21.]], grad_fn=<AddBackward0>)

Gradients (dy/dx):
 tensor([[0.5000, 1.0000],
        [1.5000, 2.0000]])


## 2. Defining an nn.Module

Here, we define a simple neural network architecture by subclassing `nn.Module`. We specify the layers in `__init__` and dictate how data flows through them in the `forward` method.

In [11]:
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNet()
print(model)

# Test with dummy data
dummy_input = torch.randn(4, 10) # Batch size of 4, input features 10
output = model(dummy_input)
print("\nOutput shape:", output.shape)


SimpleNet(
  (fc1): Linear(in_features=10, out_features=5, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=5, out_features=2, bias=True)
)

Output shape: torch.Size([4, 2])


## 3. DataLoader and Optimizer Loop

This section demonstrates the standard PyTorch training pipeline: wrapping data in a `TensorDataset` and `DataLoader` for batching, defining a loss function, and updating model weights iteratively using an optimizer.

In [10]:
# Create dummy data
inputs = torch.randn(100, 10)
targets = torch.randint(0, 2, (100,)) # Binary classification

# Wrap in DataLoader
dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 5
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_inputs, batch_targets in dataloader:
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(batch_inputs)
        
        # Compute loss
        loss = criterion(predictions, batch_targets)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(dataloader):.4f}")


Epoch 1/5, Loss: 0.7153
Epoch 2/5, Loss: 0.7111
Epoch 3/5, Loss: 0.7039
Epoch 4/5, Loss: 0.7079
Epoch 5/5, Loss: 0.7027
